In [1]:
# Import required libraries
import pandas as pd
import numpy as np 
import math 
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import os, glob, re
from itertools import cycle
from pathlib import Path
from PIL import Image

In [ ]:
# 0) OPTIONAL  ▸ set to 1031  or  [1031, 1142] (Participant Identifier)  to load only these, if it should load all Participants: set it to None
participants_to_load = None

# 1) WHAT TO LOAD – just edit this list
sessions_to_load = [1]          # e.g. [1, 2, 3...]

# 2) WHERE YOUR (turn-annotated) CSVs LIVE
folder = r"C:/Users/Marcel/Desktop/Study Project/VR Data/Data with Turns"

# LOAD ONLY THE DESIRED SESSION FILES
pattern = re.compile(r'(\d{4})_(\d).*\.csv$')   # matches 1234_1.csv
dfs = []                                        # will collect DataFrames here

for filepath in glob.glob(os.path.join(folder, "*.csv")):
    m = pattern.search(os.path.basename(filepath))
    if not m:
        continue
    pid, sess = map(int, m.groups())

    # --- participant filter -------------------------------------------
    if participants_to_load is not None:
        if isinstance(participants_to_load, (list, tuple, set)):
            if pid not in participants_to_load:
                continue
        else:                      # single int
            if pid != participants_to_load:
                continue

    # --- session filter -----------------------------------------------
    if sess not in sessions_to_load:
        continue

    # --- load ----------------------------------------------------------
    df = pd.read_csv(filepath, sep=",")
    dfs.append(df)

# SUMMARY OF WHAT GOT LOADED
print(f"Loaded {len(dfs)} file(s) for session(s) {sessions_to_load} "
      f"{'and participants ' + str(participants_to_load) if participants_to_load else '(all participants)'}."
)

Loaded 1 file(s) for session(s) [1] and participants 1031.


In [4]:
## Plotting function with City Map Image roughly aligned
# ── 1)  load the PNG once at import time --------------------------------
MAP_PATH = Path(r"C:/Users/Marcel/Desktop/Study Project/VR Data/city_map.png")
IMG      = Image.open(MAP_PATH)         
IMG_W, IMG_H = IMG.size                 

# ── 2)  fixed alignment constants ------------------
OFFSET_X = -490      
OFFSET_Y =  400      
SCALE    = 0.79      

def plot_path(
    df,
    *,
    highlight=None,                   
    title="Path with map overlay",
    width=1400,
    height=1000,
    opacity=1.0,
    show=True                          
):
    """
    Draw the participant trajectory with the fixed city_map PNG beneath.
    """

    # ------------------------------------------------------------------
    # 1)  base scatter
    fig = px.scatter(
        df,
        x="playerBodyPosition.x",
        y="playerBodyPosition.z",
        color=df.index,
        hover_data={"total_time": ':.2f'},
        color_continuous_scale="Viridis",
        labels={"playerBodyPosition.x": "Body X",
                "playerBodyPosition.z": "Body Z",
                "total_time": "Time",
                "color": "Index"},
        title=title,
        width=width,
        height=height,
    )
    fig.update_layout(
        yaxis=dict(scaleanchor="x", scaleratio=1),
        dragmode="pan",
        margin=dict(l=0, r=0, t=50, b=0),
    )

    # ------------------------------------------------------------------
    # 2)  map background
    fig.add_layout_image(
        dict(
            source=IMG,
            xref="x",
            yref="y",
            x=OFFSET_X,
            y=OFFSET_Y,
            sizex=IMG_W / SCALE,
            sizey=IMG_H / SCALE,
            sizing="stretch",
            layer="below",
            opacity=opacity,
        )
    )
    fig.update_xaxes(range=[OFFSET_X, OFFSET_X + IMG_W / SCALE])
    fig.update_yaxes(range=[OFFSET_Y, OFFSET_Y + IMG_H / SCALE])

    # ------------------------------------------------------------------
    # 3)  optional highlight indices
    if highlight is not None and len(highlight):
        sel = df.loc[highlight]
        cd = np.column_stack([sel.index, sel["total_time"]])
        fig.add_trace(
            go.Scatter(
                x=sel["playerBodyPosition.x"],
                y=sel["playerBodyPosition.z"],
                mode="markers",
                marker=dict(color="red", size=12, symbol="circle"),
                name="highlight",
                customdata=cd,
                hovertemplate=(
                    "index %{customdata[0]}<br>"
                    "time %{customdata[1]:.2f}s<br>"
                    "X =%{x:.2f}<br>Z =%{y:.2f}"
                    "<extra></extra>"
                ),
            )
        )

    # ------------------------------------------------------------------
    # 4)  display only if requested
    if show:
        fig.show(
            renderer="browser",
            config={
                "responsive": True,
                "scrollZoom": True,  # keeps wheel-zoom enabled
                "displaylogo": False,
            },
        )

    return fig

In [5]:
# ---------------------------------------------------------------------
# 3) ── PLOTLY OVERLAY HELPERS 
from itertools import cycle
import plotly.express as px
import plotly.graph_objects as go

def add_new_turn_markers(fig, df):
    """Red circles + order numbers for rows where entry_nr == 1."""
    turns = df[df["entry_nr"] == 1]
    if turns.empty:
        return

    # red ring
    fig.add_trace(
        go.Scatter(
            x=turns["playerBodyPosition.x"],
            y=turns["playerBodyPosition.z"],
            mode="markers",
            marker=dict(symbol="circle-dot",
                        size=30,
                        color="rgba(255,0,0,0)",
                        line=dict(color="red", width=4)),
            showlegend=False,
            hovertemplate="First entry #%{text}<extra></extra>",
            text=[str(i) for i in range(1, len(turns) + 1)]
        )
    )

    # index number (offset a touch up-right)
    fig.add_trace(
        go.Scatter(
            x=turns["playerBodyPosition.x"] + 2,
            y=turns["playerBodyPosition.z"] + 2,
            mode="text",
            text=[str(i) for i in range(1, len(turns) + 1)],
            textfont=dict(color="black", size=30),
            textposition="top right",
            hoverinfo="skip",
            showlegend=False,
        )
    )

def add_repeat_markers(fig, df, size=28):
    """
    Highlight rows where the same street_id_within_participant
    is entered multiple times (entry_nr > 1). Each street gets its own colour.
    """
    rep_groups = (
        df.groupby("street_id_within_participant")
          .filter(lambda g: g["entry_nr"].max() > 1)
          .groupby("street_id_within_participant")
    )
    if rep_groups.ngroups == 0:
        return

    palette = cycle(px.colors.qualitative.Safe)     
    for street_id, g in rep_groups:
        colour = next(palette)
        fig.add_trace(
            go.Scatter(
                x=g["playerBodyPosition.x"],
                y=g["playerBodyPosition.z"],
                mode="markers",
                marker=dict(
                    color=colour,
                    symbol="star-triangle-up",
                    size=30,                         
                    line=dict(width=2, color="black") 
                ),
                name=f"street {street_id}",
                hovertemplate=f"street {street_id}<extra></extra>",
            )
        )

# ---------------------------------------------------------------------
# 3½) ── helper to show with scroll-zoom -------------------------------
def show_fig(fig):
    fig.show(
        renderer="browser",
        config={
            "responsive": True,
            "scrollZoom": True,  
            "displaylogo": False
        }
    )

In [6]:
# ---------------------------------------------------------------------
# ── MAIN INTERACTIVE LOOP -------------------------------
for df in dfs:                               # one DataFrame per participant
    pid = df["SubjectID"].iloc[0]

    # 1️⃣  FIRST VIEW – numbered new turns ----------------------------
    fig1 = plot_path(df, title=f"Participant {pid} - all turns", show=False)
    add_new_turn_markers(fig1, df)
    show_fig(fig1)

    key = input("[ENTER] show repeated-street view   |   [SPACE] next ▶ ").strip()

    # 2️⃣  SECOND VIEW – repeated-street highlights -------------------
    if key == "":                            # ENTER ⇒ same participant
        fig2 = plot_path(df,
                         title=f"Participant {pid} – repeated street entries",
                         show=False)
        add_repeat_markers(fig2, df, size=28)  # bigger diamonds
        show_fig(fig2)
        input("Press [SPACE] to continue ▶ ")